# Band Power Feature Extraction

**Dataset**: MOABB BNCI2014-001 (Motor Imagery)  
**Channels**: 22 channels  
**Sampling rate**: 250 Hz  
**Subject**: 1

---

## Overview

We compute alpha and beta band power features for each channel using Welch's method, producing 44 features per trial.

## What this notebook does

Computes power spectral density for each channel and trial, integrates over alpha (8-13 Hz) and beta (13-30 Hz) bands.

## What you should expect to see

- Heatmap of the feature matrix showing power values across trials and features
- Bar chart of mean power per channel (alpha + beta combined)
- 44 features total (22 channels x 2 bands)

## Key parameters

| Parameter | Value |
| --- | --- |
| fmin | 8 |
| fmax | 32 |
| n_classes | 2 |
| FS | 250 |
| bands | alpha (8-13), beta (13-30) |


## 1. Install dependencies


In [ ]:
!pip install moabb mne scipy numpy plotly scikit-learn


## 2. Load MOABB dataset

MOABB downloads data automatically on first use (~44 MB).


In [ ]:
from moabb.datasets import BNCI2014_001
from moabb.paradigms import MotorImagery
import numpy as np

dataset = BNCI2014_001()
paradigm = MotorImagery(n_classes=2, fmin=8, fmax=32)
X, labels, meta = paradigm.get_data(dataset=dataset, subjects=[1])

print(f'X shape: {X.shape}')
print(f'Labels: {np.unique(labels)}')
print(f'Trials: {len(labels)}')


In [ ]:
mask = (labels == 'left_hand') | (labels == 'right_hand')
X = X[mask]
labels = labels[mask]

print(f'After filtering - X shape: {X.shape}')
print(f'Labels: {np.unique(labels)}')


## 3. Explore the data


In [ ]:
n_trials, n_channels, n_samples = X.shape
print(f'Trials: {n_trials}')
print(f'Channels: {n_channels}')
print(f'Samples per trial: {n_samples}')
print(f'Trial duration: {n_samples/250:.2f} s')


## 4. Compute band power features


In [ ]:
from scipy.signal import welch

FS = 250
BANDS = [(8, 13, 'alpha'), (13, 30, 'beta')]

features = np.zeros((n_trials, n_channels * len(BANDS)))
for trial in range(n_trials):
    for ch in range(n_channels):
        freqs, psd = welch(X[trial, ch, :], fs=FS, nperseg=256)
        for b_idx, (fmin, fmax, bname) in enumerate(BANDS):
            mask_f = (freqs >= fmin) & (freqs <= fmax)
            features[trial, ch * len(BANDS) + b_idx] = np.trapezoid(psd[mask_f], freqs[mask_f])

print(f'Feature matrix shape: {features.shape}')

mean_power = np.zeros(n_channels)
for ch in range(n_channels):
    alpha_idx = ch * len(BANDS) + 0
    beta_idx = ch * len(BANDS) + 1
    mean_power[ch] = np.mean(features[:, alpha_idx]) + np.mean(features[:, beta_idx])

print(f'Mean power per channel computed for {n_channels} channels')


## 5. Interactive plot

**What to look for:**

- The heatmap shows variation in power across trials and features
- Some channels have higher mean power than others
- Alpha and beta bands capture motor imagery related activity


In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = make_subplots(rows=2, cols=1, subplot_titles=(
    'Feature Matrix (first 50 trials x 44 features)',
    'Mean Power per Channel'))

fig.add_trace(go.Heatmap(z=features[:50, :], colorscale='Viridis', name='Features', showscale=True), row=1, col=1)
fig.add_trace(go.Bar(x=list(range(n_channels)), y=mean_power, marker_color='steelblue', name='Mean Power'), row=2, col=1)

fig.update_xaxes(title_text='Feature (channel x band)', row=1, col=1)
fig.update_yaxes(title_text='Trial', row=1, col=1)
fig.update_xaxes(title_text='Channel', row=2, col=1)
fig.update_yaxes(title_text='Mean Power (alpha + beta)', row=2, col=1)
fig.update_layout(height=800, showlegend=False, title_text='Band Power Features for ML')
fig.show()


## What did we learn?

- Band power features are a simple yet effective representation for EEG classification
- 44 features (22 channels x 2 bands) capture spectral information in motor imagery bands
- Welch's method provides a robust PSD estimate for feature extraction
